In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors


import torch
from torch.utils.data import TensorDataset, DataLoader

# convert image arrays to float32 tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

# convert age labels to float32 tensors
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# tensor shapes and types
print("Tensor Shapes:")
print(f"  X_train_tensor: {X_train_tensor.shape}, dtype: {X_train_tensor.dtype}")
print(f"  X_test_tensor:  {X_test_tensor.shape}, dtype: {X_test_tensor.dtype}")
print(f"  y_train_tensor: {y_train_tensor.shape}, dtype: {y_train_tensor.dtype}")
print(f"  y_test_tensor:  {y_test_tensor.shape}, dtype: {y_test_tensor.dtype}")

In [ ]:
# 2. Create TensorDataset objects

# training dataset: each training image with its correspnding age
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# testing dataset the same
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# check datasets created correctly
print(f"Training dataset size: {len(train_dataset)} samples")
print(f"Testing dataset size:  {len(test_dataset)} samples")

# one sample
sample_image, sample_age = train_dataset[0]
print(f"\nSample from training dataset:")
print(f"  Image shape: {sample_image.shape}")
print(f"  Age label:   {sample_age.item():.1f} years")

In [ ]:
# 3. Create DataLoaders

batch_size = 32

# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True  # randomize training data order
)


# DataLoader for test/validation data
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False  # keep order consistent for testing
)

print(f"DataLoaders created with batch_size={batch_size}")
print(f"  Training batches per epoch: {len(train_loader)}")
print(f"  Testing batches: {len(test_loader)}")


In [ ]:
# 4. Print shape of one batch

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# 5. Display sample images
import matplotlib.pyplot as plt

# batch of images and ages
images_batch, ages_batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

# show the first 8 images from the batch
for idx in range(8):
    # convert from (C, H, W) to (H, W, C) for matplotlib
    img = images_batch[idx].permute(1, 2, 0).numpy()

    age = ages_batch[idx].item()

    # show the image
    axes[idx].imshow(img)
    axes[idx].set_title(f"Age: {age:.0f} years", fontsize=12)
    axes[idx].axis('off')

plt.suptitle("Display sample images", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()



In [ ]:
# Task 1: Write your model class here:


import torch.nn as nn

class NN4Layer(nn.Module):

    def __init__(self, input_dim):
        super(NN4Layer, self).__init__()

        # input_dim = num of features
        self.layer1 = nn.Linear(input_dim, 512)

        # from 512 to 256
        self.layer2 = nn.Linear(512, 256)

        # from 256 to 128
        self.layer3 = nn.Linear(256, 128)

        # outputs single value: the predicted age
        # NO activation here bc it is regression
        self.layer4 = nn.Linear(128, 1)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)

        z1 = self.layer1(x)   # linear transformation
        a1 = self.relu(z1)     # Non-linear activation

        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        z3 = self.layer3(a2)
        a3 = self.relu(z3)

        # NO activation for regreion output
        z4 = self.layer4(a3)

        return x


In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):

    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Move batch to the selected device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    return avg_loss

print("defined successfully!")

In [ ]:
# Task 3: Write your validation loop here:


def validate(model, criterion, test_loader, device):

    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    total_mae = 0.0  # Mean Absolute Error (more interpretable than MSE)
    total_samples = 0

    with torch.no_grad():  # Disable gradient computation
        for X_batch, y_batch in test_loader:
            # Move batch to the selected device
            X_batch = X_batch.to(device)
            y_batch = y_batch.view(-1, 1).to(device)

            # Forward pass
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Binary predictions
            predicted = (outputs > 0.5).float()

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)


    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

print("defined successfully!")

In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


sample_img = X_train_tensor[0]  # Shape: (C, H, W)
input_dim = sample_img.shape[0] * sample_img.shape[1] * sample_img.shape[2]
print(f"Input dimension: {input_dim} (flattened image size)")
print(f"  - {sample_img.shape[0]} channels × {sample_img.shape[1]} height × {sample_img.shape[2]} width")


model = NN4Layer(input_dim).to(device)
criterion = nn.MSELoss()
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print("MODEL ARCHITECTURE")
print(model)

In [ ]:
# Task 5: Start training for 20 epochs:
# Run Training
num_epochs = 20

train_losses = []
val_losses = []
val_maes = []
print("Starting Training...")


for epoch in range(num_epochs):

    # Train for one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)
    train_losses.append(train_loss)

    # Validate on test set
    val_loss, val_mae = validate(model, criterion, test_loader, device)
    val_losses.append(val_loss)
    val_maes.append(val_mae)

    if (epoch + 1) % 5 == 0:
       print(f'Epoch [{epoch+1}/{20}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val MAES: {val_maes:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training vs Validation Loss
ax1 = axes[0]
epochs_range = range(1, num_epochs + 1)

ax1.plot(epochs_range, train_losses, 'b-', linewidth=2, label='Training Loss', marker='o', markersize=5)
ax1.plot(epochs_range, val_losses, 'r-', linewidth=2, label='Validation Loss', marker='s', markersize=5)

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (MSE)', fontsize=12)
ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: validation MAE
ax2 = axes[1]

ax2.plot(epochs_range, val_maes, 'g-', linewidth=2, label='Validation MAE', marker='^', markersize=5)

ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Mean Absolute Error (Years)', fontsize=12)
ax2.set_title('Validation MAE Over Epochs', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

ax2.axhline(y=val_maes[-1], color='gray', linestyle='--', alpha=0.5,
            label=f'Final MAE: {val_maes[-1]:.2f} years')

plt.tight_layout()
plt.show()